## **Title**

In [1]:
#TASK 3: Multimodal ML (Images + Tabular Data)
#Objective:
#Predict house prices using both:
#Images (CNN features)
#Tabular data

## **CNN Feature Extractor**

In [36]:
# Import PyTorch
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error

from PIL import Image
# Load pre-trained CNN (ResNet18)
cnn = models.resnet18(pretrained=True)

# Remove final classification layer (we only need features)
cnn.fc = nn.Identity()
cnn = models.resnet18(weights="IMAGENET1K_V1")  # updated safe version
cnn.fc = nn.Identity()

## **Tabular Data**

In [108]:


# Load housing data
df = pd.read_csv("housing.csv")

# Split features and target
X_tab = df.drop("price", axis=1)
y = df["price"]

## **Image Directory**

In [109]:
img_dir="C:/Users/Zaman Computer/Downloads/New folder/images"

## **Dataset Class**

In [110]:
class HouseDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        # image path
        img_path = f"{self.img_dir}/{self.df.iloc[idx]['image_name']}"

        # load image
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        # tabular data
        
        tabular = torch.tensor(
                    self.df.iloc[idx][["f1","f2","f3"]].astype(float).values,
                    dtype=torch.float32
        )
        # target
        label = torch.tensor(self.df.iloc[idx]["price"], dtype=torch.float32)

        return image, tabular, label

## **Data Loader**

In [111]:
dataset = HouseDataset(df, img_dir)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

## **CNN Model**

In [112]:
cnn = models.resnet18(weights="IMAGENET1K_V1")
cnn.fc = nn.Identity()


## **Multimodal Model**

In [113]:
class MultiModalModel(nn.Module):
    def __init__(self, tabular_dim):
        super().__init__()

        self.cnn = cnn
        self.fc_tab = nn.Linear(tabular_dim, 64)
        self.fc_final = nn.Linear(512 + 64, 1)

    def forward(self, image, tabular):
        img_feat = self.cnn(image)
        tab_feat = self.fc_tab(tabular)

        combined = torch.cat((img_feat, tab_feat), dim=1)
        return self.fc_final(combined)

model = MultiModalModel(tabular_dim=3)

## **Loss + Optimizer**


In [114]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## **Training Loop**

In [115]:
for epoch in range(5):
    for images, tabular, labels in dataloader:

        optimizer.zero_grad()

        outputs = model(images, tabular)

        loss = criterion(outputs, labels.unsqueeze(1))

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} Loss: {loss.item()}")

Epoch 1 Loss: 251781545984.0
Epoch 2 Loss: 40142528512.0
Epoch 3 Loss: 101099782144.0
Epoch 4 Loss: 151210573824.0
Epoch 5 Loss: 160987955200.0


## **Final Insights**

In [116]:

### **Combining image + tabular improves prediction
### **CNN extracts useful visual features
#Multimodal learning is powerful but complex